<a href="https://colab.research.google.com/github/thetallguy14/flyrank-ml-internship/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
## Unit of Analysis

One row represents the daily performance of a single content item (`content_hash_id`) for a specific client (`client_hash_id`) on a single reporting date (`report_date`).

Each row contains historical Google Search Console (GSC) and Google Analytics 4 (GA4) metrics that describe the search visibility and user engagement of that content on that day.

## Time Window

This notebook uses the **March 2026** partition (`month = '2026-03'`) to develop and verify the data contract. The final month (June 2026) is intentionally excluded because it should remain an unseen evaluation period.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
## Features

The following features are used because they are available before making an optimization decision:

- gsc_impressions
- gsc_avg_position
- ctr (derived from historical clicks and impressions)
- ga4_sessions
- engagement_rate (derived from engaged sessions and total sessions)

## Label

This warehouse slice does not include a dedicated opportunity score. For the leakage demonstration, **gsc_clicks** is used as a proxy label.

## Context Fields

These columns identify the observation but are not used directly for prediction:

- report_date
- client_hash_id
- content_hash_id
- month

## Excluded Fields

The following fields are excluded:

- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

Reason:
These AI referral metrics are sparse in this data slice and are not required for the basic feature engineering exercise.

Future information and any variables computed after the prediction date are also excluded to avoid data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.register("fact_content_daily_performance", df)

In [ ]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS records
FROM fact_content_daily_performance
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,records


In [ ]:
import numpy as np

# CTR
df["ctr"] = (
    df["gsc_clicks"].fillna(0)
    / df["gsc_impressions"].replace(0, np.nan)
).fillna(0)

# Engagement Rate
df["engagement_rate"] = (
    df["ga4_engaged_sessions"].fillna(0)
    / df["ga4_sessions"].replace(0, np.nan)
).fillna(0)

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,ctr,engagement_rate
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,0.0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,0.0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.008,0.0
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,0.0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,0.000,0.0


In [ ]:
features = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ctr",
        "ga4_sessions",
        "engagement_rate"
    ]
]

features.head()

,gsc_impressions,gsc_avg_position,ctr,ga4_sessions,engagement_rate
0,20,3.350000,0.000,<NA>,0.0
1,1,0.000000,0.000,<NA>,0.0
2,125,4.928000,0.008,<NA>,0.0
3,7,4.000000,0.000,<NA>,0.0
4,11,2.272727,0.000,<NA>,0.0


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = features
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest R²:", r2_score(y_test, pred))

Honest R²: 0.9939557289348432


In [ ]:
features["leak"] = df["label"]

X = features

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaked R²:", r2_score(y_test, pred))

/tmp/ipykernel_3042/4084441378.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["leak"] = df["label"]


In [ ]:
features = features.drop(columns=["leak"])

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
## Data Limitations

- This notebook analyzes only the March 2026 partition and does not capture seasonal trends across multiple months.
- Some GA4 metrics contain missing values because not every client has GA4 connected.
- The warehouse stores observed historical behavior and cannot explain why user behavior changes.
- The data contains anonymized client and content identifiers, so results cannot be linked back to specific websites.
- The analysis provides decision support and ranking signals rather than causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.